<div style="text-align: center;">
    <a href="https://www.hi-paris.fr/">
        <img border="0" src="https://www.hi-paris.fr/wp-content/uploads/2020/09/logo-hi-paris-retina.png" width="50%"></a>
    <img src="https://upload.wikimedia.org/wikipedia/commons/0/0a/Detailed_Power_usage.png" width="50%">
</div>

# Audensiel - REFIT challenge

<i> Team 27 </i><br/>
<i> Ruben BUENO </i><br/>
<i> Toscane CARRO </i><br/>
<i> Gabriel FRANCOIS </i><br/>
<i> Clément GAUBERT </i><br/>
<i> Alice LATASTE </i><br/>
<i> Soline MIGNOT </i>

## Introduction

This challenge is based on the [REFIT](http://dx.doi.org/10.1038/sdata.2016.122) dataset, which contains electricity consumption data (both global and individual appliances) for different houses :

- The R&D department of AUDENSIEL has modified these datasets for the purpose of their EcoSmartGrid project and graciously provided them for use on this challenge.
- This challenge is an anomaly detection task applied to time series.  
- The goal of this task is to be able to detect consumption anomalies which indicate an individual equipment is malfunctioning, via the monitoring of the house's electricity meter.


### Anomaly Detection:

Anomaly detection refers to the task of identifying observations that significantly deviate from the majority of the data. These anomalies (or outliers) are usually:

- Rare
- Difficult to label
- Different in nature from normal patterns

Most anomaly detection methods follow a common modeling principle: first learn a representation of normal data, then compute a score that quantifies how much each observation deviates from this normal behavior.

### Baseline selected for the starting kit : Isolation Forest

Isolation Forest is an unsupervised anomaly detection method that identifies anomalies by isolating observations rather than modeling normal behavior.  
It builds random trees and measures how quickly a point is isolated: **anomalies require fewer splits**, while normal points lie deeper in the trees.



# Exploratory data analysis

The data comes from four homes (H2, H3, H9, H15) from the REFIT database. The datasets have been adapted to annotate instances showing abnormal consumption related to the refrigerator/freezer. The recordings are synchronized to an 8-second time step.
### Data Description
The files include the following column families:   
- `unix (int64)`: timestamp in Unix format (seconds since 1970-01-01).  
- `time (string)`: date-time in readable format (e.g., 2014-03-21 00:00:00).  
- `Global signal` (variable usable in production)  
- `aggregate (float64)`: total household consumption (instantaneous power, in Watts).  
- `Sub-measurements per appliance` (reference data)  
Some columns may correspond to consumption per appliance (float64), including: fridge freezer, washing machine, microwave, dishwasher, washer dryer, electric space heater, television, audio system, kettle.  
These columns are provided for reference and analysis purposes and must not be used as input variables for the model in the context of this challenge.  
- Partial aggregate (reference data)  
- `agg_4p (float64)`: exact aggregate of four appliances (washing machine, microwave, dishwasher, kettle).  
- Annotations (targets)  
- `anomaly (int64)`: label indicating the state of the refrigerator/freezer (normal or abnormal) according to the coding used in the annotated files.  



In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data using the ingestion program
from ingestion_program.ingestion import get_train_data
X_df, y_df = get_train_data("dev_phase/input_data")

df = pd.concat([X_df, y_df], axis=1)

df['time'] = pd.to_datetime(df['unix'], unit='s')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())

ModuleNotFoundError: No module named 'ingestion_program'

Check Class Imbalance

In [ ]:
counts = df['anomaly'].value_counts()
percentage = counts[1] / len(df) * 100

print(f"Normal samples:   {counts[0]}")
print(f"Anomaly samples:  {counts[1]}")
print(f"Anomaly Ratio:    {percentage:.2f}%")

# Visualize the imbalance
plt.figure(figsize=(6, 4))
sns.countplot(x='anomaly', data=df)
plt.title("Class Distribution (0=Normal, 1=Anomaly)")
plt.show()

Visualize Time Series

In [ ]:
# Filter for a specific house (e.g., House 3) and sort by time
house_id = 3
subset = df[df['house_id'] == house_id].sort_values('unix')

plt.figure(figsize=(15, 5))

# Plot the aggregate energy consumption
plt.plot(subset['time'], subset['aggregate'], label='Energy Consumption', color='blue', alpha=0.6)

# Overlay the anomalies in red
anomalies = subset[subset['anomaly'] == 1]
plt.scatter(anomalies['time'], anomalies['aggregate'], color='red', label='Anomaly', s=30, zorder=5)

plt.title(f"Energy Consumption Profile - House {house_id}")
plt.xlabel("Time")
plt.ylabel("Aggregate Energy")
plt.legend()
plt.show()

# Challenge evaluation

Submissions will be evaluated using a weighted scoring system that balances model efficiency, storage, performance, and generalization. The final score is composed of three equally weighted components: model size, performance on the public test set, and performance on the private test set.

For the model size, models <=20 MB receive 1, while larger models are penalized linearly, reaching zero at 40MB.

For model performance, for each evaluation set (public and private), performance is measured as a 50/50 split between recall and prediction time.


# Submission format



## The submission file

The input data are stored in a dataframe. To go from a dataframe to a numpy array we will use a scikit-learn column transformer. The first example we will write will just consist in selecting a subset of columns we want to work with. 


In [ ]:
from sklearn.ensemble import IsolationForest


# The submission here should simply be a function that returns a model
# compatible with scikit-learn API
def get_model():
    return IsolationForest() # will give labels like -1 or 1, not 0 or 1, so might be bad depending on what was done for the labeling part of the data

## Local testing pipeline

Here you can show how the model will be used to generate predictions on the test set, and how the evaluation will be performed.

In [ ]:
model = get_model()
X_train, y_train = get_train_data("dev_phase/input_data")
model.fit(X_train, y_train)

X_test = pd.read_csv("dev_phase/input_data/test/test_features.csv")
from ingestion_program.ingestion import evaluate_model
y_test = evaluate_model(model, X_test)

from scoring_program.scoring import compute_accuracy
print("Accuracy on test set:", compute_accuracy(y_test, pd.read_csv("dev_phase/input_data/test/test_labels.csv")))

Accuracy on test set: 0.95


/home/tom/.local/miniconda/lib/python3.12/site-packages/sklearn/base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


## Submission

To submit your code, you can refer to the actual challenge.